# GPQA Diamond — Phase 8 Part B (Fixed Model Loading)

Standalone notebook. Runs GPQA Diamond inference + spectral hallucination detection.

**Model:** `Qwen/Qwen2.5-72B-Instruct` (4-bit NF4, bfloat16)  
**Dataset:** GPQA Diamond (~198 problems)  
**Requires:** Colab A100 (40 GB)

### Fixes vs Phase 8 original
- `torch_dtype=torch.bfloat16` — was `dtype=torch.float16`, an **invalid** kwarg silently ignored by `from_pretrained`  
- `bnb_4bit_compute_dtype=torch.bfloat16` — Qwen2.5 is bfloat16-native; float16 caused instability  
- `attn_implementation="eager"` — avoids flash-attention import error on Colab  
- `trust_remote_code=False` — not needed for Qwen2.5 in transformers>=4.37

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers>=4.40', 'datasets',
                'accelerate', 'scipy', 'bitsandbytes'], check=True)

import os, gc, re, pickle, itertools, hashlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from scipy.linalg import eigh
from scipy.signal import stft as scipy_stft
from tqdm import tqdm
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5),
                     'font.size': 11, 'axes.titlesize': 13})
print('Imports OK.')

Imports OK.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HuggingFace login OK.')
except Exception as e:
    print(f'HF login skipped: {e}')

GPQA_BASE_DIR     = '/content/drive/MyDrive/epr_spectral_gpqa_72b/'
GPQA_RUN_DIR      = GPQA_BASE_DIR + 'Qwen2.5-72B-Instruct__gpqa_T1.0/'
GPQA_CACHE_PATH   = GPQA_RUN_DIR  + 'inference_cache.pkl'
GPQA_RESULTS_PATH = GPQA_RUN_DIR  + 'gpqa72b_results.pkl'
GPQA_PLOT_DIR     = GPQA_BASE_DIR
os.makedirs(GPQA_RUN_DIR, exist_ok=True)

MODEL_ID  = 'Qwen/Qwen2.5-72B-Instruct'
TEMP      = 1.0
MAX_NEW   = 1024
QUANTIZE  = True

PRIOR_GPQA_7B = 0.654

print(f'Run dir:  {GPQA_RUN_DIR}')
print(f'Cache:    {GPQA_CACHE_PATH}')
_done = os.path.exists(GPQA_RESULTS_PATH)
print(f'Status:   {"DONE (results exist)" if _done else "pending inference"}')
if os.path.exists(GPQA_CACHE_PATH):
    _c = pickle.load(open(GPQA_CACHE_PATH, 'rb'))
    _n = sum(1 for v in _c.values() if isinstance(v, dict) and v.get('done'))
    print(f'Cache:    {_n}/{len(_c)} entries done')

Mounted at /content/drive
HuggingFace login OK.
Run dir:  /content/drive/MyDrive/epr_spectral_gpqa_72b/Qwen2.5-72B-Instruct__gpqa_T1.0/
Cache:    /content/drive/MyDrive/epr_spectral_gpqa_72b/Qwen2.5-72B-Instruct__gpqa_T1.0/inference_cache.pkl
Status:   pending inference


In [ ]:
# ── I/O ────────────────────────────────────────────────────────────────────────
def load_cache(path):
    return pickle.load(open(path, 'rb')) if os.path.exists(path) else {}

def save_cache(obj, path):
    pickle.dump(obj, open(path, 'wb'))

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Feature extraction ─────────────────────────────────────────────────────────
def compute_spectral_features(ents, min_len=8):
    e = np.array(ents, dtype=float); N = len(e)
    if N < min_len: return None
    e_ac = e - e.mean()
    fft_vals = np.fft.rfft(e_ac)
    psd      = np.abs(fft_vals) ** 2
    freqs    = np.fft.rfftfreq(N)
    psd_sum  = psd.sum() + 1e-12
    psd_norm = psd / psd_sum
    spec_ent  = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
    low_mask  = (freqs > 0.0) & (freqs <= 0.10)
    high_mask = (freqs >= 0.40) & (freqs <= 0.50)
    low_power  = psd[low_mask].sum()  / psd_sum
    high_power = psd[high_mask].sum() / psd_sum
    hl_ratio   = high_power / (low_power + 1e-12)
    ac_mask  = freqs > 0
    dom_freq = float(freqs[ac_mask][np.argmax(psd[ac_mask])]) if ac_mask.sum() > 0 else 0.0
    centroid = float(np.sum(freqs[ac_mask] * psd_norm[ac_mask]) /
                     (psd_norm[ac_mask].sum() + 1e-12)) if ac_mask.sum() > 0 else 0.0
    return {'spectral_entropy': float(spec_ent), 'low_band_power': float(low_power),
            'high_band_power': float(high_power), 'hl_ratio': float(hl_ratio),
            'dominant_freq': dom_freq, 'spectral_centroid': centroid}

def compute_stft_features(ents, nperseg=16, noverlap=8, min_len=32):
    e = np.array(ents, dtype=float)
    if len(e) < min_len:
        return {'stft_max_high_power': 0.0, 'stft_spectral_entropy': 0.0}
    e_ac = e - e.mean()
    f, t, Zxx = scipy_stft(e_ac, nperseg=nperseg, noverlap=noverlap)
    psd = np.abs(Zxx) ** 2
    high_mask = f >= 0.40
    if high_mask.sum() > 0 and psd.shape[1] > 0:
        high_frac      = psd[high_mask].sum(0) / (psd.sum(0) + 1e-12)
        max_local_high = float(high_frac.max())
    else:
        max_local_high = 0.0
    psd_n     = psd / (psd.sum(0, keepdims=True) + 1e-12)
    frame_ent = -np.sum(psd_n * np.log(psd_n + 1e-12), axis=0)
    stft_ent  = float(frame_ent.mean()) if len(frame_ent) > 0 else 0.0
    return {'stft_max_high_power': max_local_high, 'stft_spectral_entropy': stft_ent}

def compute_time_domain(ents, tail_frac=0.20, sw_window=16, sw_step=1):
    e = np.array(ents, dtype=float)
    W = max(1, int(len(e) * tail_frac))
    rpdi = float(e[-W:].mean() / (e.mean() + 1e-12))
    if len(e) >= sw_window:
        sw_vars     = [np.var(e[i:i+sw_window])
                       for i in range(0, len(e)-sw_window+1, sw_step)]
        sw_var_peak = float(np.max(sw_vars))
    else:
        sw_var_peak = float(np.var(e))
    return {'rpdi': rpdi, 'sw_var_peak': sw_var_peak}

def extract_all_features(ents):
    e = np.array(ents, dtype=float)
    result = {'epr': float(e.mean()), 'trace_length': float(len(e))}
    gf = compute_spectral_features(ents)
    if gf is None: return None
    result.update(gf)
    result.update(compute_stft_features(ents))
    result.update(compute_time_domain(ents))
    return result

def sw_var_peak_with_window(ents, sw_window, sw_step=1):
    e = np.array(ents, dtype=float)
    if len(e) >= sw_window:
        return float(np.max([np.var(e[i:i+sw_window])
                              for i in range(0, len(e)-sw_window+1, sw_step)]))
    return float(np.var(e))

FEAT_NAMES = ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power',
              'hl_ratio', 'dominant_freq', 'spectral_centroid',
              'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak']

# ── Statistics ─────────────────────────────────────────────────────────────────
def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-12)

def boot_auc(y, scores, n=1000):
    y, s = np.array(y), np.array(scores)
    if len(np.unique(y)) < 2 or np.std(s) < 1e-8:
        return float('nan'), float('nan'), float('nan')
    base = roc_auc_score(y, s)
    rng  = np.random.default_rng(42)
    boots = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        boots.append(roc_auc_score(y[idx], s[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (base, base)
    return base, lo, hi

def nadler_fuse(*views):
    X = np.column_stack(views); n, k = X.shape
    C = np.cov(X.T)
    if C.ndim == 0: C = np.array([[float(C)]])
    try:
        off = C - np.diag(np.diag(C))
        rs, cs = off.sum(1), off.sum(0)
        M      = np.diag(rs) @ np.linalg.pinv(C) @ np.diag(cs)
        _, vecs = eigh(M)
        w = np.abs(vecs[:, -1]); w /= w.sum() + 1e-12
    except Exception:
        w = np.ones(k) / k
    return X @ w, w

def simple_average_fusion(*views):
    X = np.column_stack(views)
    w = np.ones(X.shape[1]) / X.shape[1]
    return X @ w, w

def best_nadler_on(feats_dict, feat_names, labels_, max_size=4, label='',
                   normalize=True, compare_mean=False):
    auc_m, sign_m = {}, {}
    for n_ in feat_names:
        ap, *_ = boot_auc(labels_,  feats_dict[n_])
        an, *_ = boot_auc(labels_, -feats_dict[n_])
        if ap >= an: auc_m[n_], sign_m[n_] = ap, +1
        else:        auc_m[n_], sign_m[n_] = an, -1
    oriented = ({n_: zscore(feats_dict[n_] * sign_m[n_]) for n_ in feat_names}
                if normalize else
                {n_: feats_dict[n_] * sign_m[n_] for n_ in feat_names})
    rho = {}
    for a, b in itertools.combinations_with_replacement(feat_names, 2):
        r, _ = spearmanr(oriented[a], oriented[b])
        rho[(a, b)] = rho[(b, a)] = r
    info  = [n_ for n_ in feat_names if auc_m[n_] > 0.50]
    total = sum(sum(1 for _ in itertools.combinations(info, sz))
                for sz in range(2, min(len(info)+1, max_size+1)))
    print(f'  [{label}] {len(feat_names)} features, {len(info)} informative, '
          f'max_size={max_size} -> {total} raw combos')
    best_a, best_lo, best_hi, best_s, best_w = 0.0, 0.0, 0.0, None, None
    checked, skipped = 0, 0
    for size in range(2, min(len(info)+1, max_size+1)):
        combos        = list(itertools.combinations(info, size))
        valid_in_size = 0
        for s in combos:
            if any(abs(rho[(a, b)]) >= 0.75 for a, b in itertools.combinations(s, 2)):
                skipped += 1; continue
            fused, w_n = nadler_fuse(*[oriented[n_] for n_ in s])
            a, lo, hi  = boot_auc(labels_, fused)
            if a > best_a:
                best_a, best_lo, best_hi, best_s, best_w = a, lo, hi, s, w_n
            checked += 1; valid_in_size += 1
        print(f'    size={size}: {len(combos)} combos, {valid_in_size} passed rho-filter, '
              f'best so far={100*best_a:.1f}%')
    print(f'  [{label}] done  checked={checked}, skipped(rho)={skipped}, '
          f'best={100*best_a:.1f}%')
    if compare_mean and best_s is not None:
        fused_avg, _ = simple_average_fusion(*[oriented[n_] for n_ in best_s])
        avg_a, avg_lo, avg_hi = boot_auc(labels_, fused_avg)
        lift = (best_a - avg_a) * 100
        print(f'\n  Nadler Lift over simple average (subset: {" + ".join(best_s)}):')
        print(f'    Nadler : {100*best_a:.1f}%  [{100*best_lo:.1f}, {100*best_hi:.1f}]')
        print(f'    Mean   : {100*avg_a:.1f}%  [{100*avg_lo:.1f}, {100*avg_hi:.1f}]')
        print(f'    Lift   : {lift:+.1f} pp')
    return best_a, best_lo, best_hi, best_s, best_w

# ── Fixed model loading ────────────────────────────────────────────────────────
def load_model(model_id, quantize_4bit=False):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    print(f'Loading tokenizer: {model_id}')
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    common_kwargs = dict(
        device_map='auto',
        torch_dtype=torch.bfloat16,    # FIX: was dtype= (invalid kwarg)
        attn_implementation='eager',   # FIX: avoids flash-attn import error
        trust_remote_code=False,       # FIX: not needed for Qwen2.5
    )
    if quantize_4bit:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,  # FIX: float16 -> bfloat16
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
        common_kwargs['quantization_config'] = bnb_cfg

    print(f'Loading model (4-bit={quantize_4bit}) ...')
    mdl = AutoModelForCausalLM.from_pretrained(model_id, **common_kwargs)
    mdl.eval()
    print(f'Loaded: {model_id}')
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU: {used:.1f} / {total:.0f} GB used')
    return mdl, tok

def fmt_prompt(tok, msg):
    try:
        return tok.apply_chat_template(
            [{'role': 'user', 'content': msg}],
            tokenize=False, add_generation_prompt=True)
    except Exception:
        return f'<|user|>\n{msg}\n<|assistant|>\n'

def token_entropies_from_scores(scores, K=15):
    ents = []
    for s in scores:
        lp   = F.log_softmax(s[0], dim=-1)
        topk = torch.topk(lp, min(K, lp.shape[-1])).values
        p    = torch.exp(topk); p = p / (p.sum() + 1e-12)
        ents.append(-(p * torch.log(p + 1e-12)).sum().item())
    return ents

def generate_full(mdl, tok, prompt_msg, temperature=1.0, K=15, max_new_tokens=512):
    prompt = fmt_prompt(tok, prompt_msg)
    inputs = tok(prompt, return_tensors='pt').to(mdl.device)
    if 'token_type_ids' in inputs:
        del inputs['token_type_ids']
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_k=50,
            output_scores=True, return_dict_in_generate=True,
            pad_token_id=tok.eos_token_id,
        )
    gen_ids   = out.sequences[0][inputs['input_ids'].shape[1]:]
    full_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    all_ents  = token_entropies_from_scores(out.scores, K)
    return full_text, all_ents

print('All helpers defined.')


All helpers defined.


In [ ]:
from datasets import load_dataset

def load_gpqa():
    ds    = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train')
    items = [dict(ds[i]) for i in range(len(ds))]
    print(f'Loaded {len(items)} GPQA Diamond problems.')
    return items

def gpqa_prompt_and_answer(row, idx):
    choices_raw = [
        row.get('Correct Answer',    ''),
        row.get('Incorrect Answer 1',''),
        row.get('Incorrect Answer 2',''),
        row.get('Incorrect Answer 3',''),
    ]
    seed = int(hashlib.md5(str(idx).encode()).hexdigest(), 16) % (2**32)
    rng  = np.random.default_rng(seed)
    order        = rng.permutation(4).tolist()
    shuffled     = [choices_raw[o] for o in order]
    correct_pos  = order.index(0)
    letters      = ['A', 'B', 'C', 'D']
    correct_letter = letters[correct_pos]
    choices_str  = '\n'.join(f'({letters[i]}) {shuffled[i]}' for i in range(4))
    question     = row.get('Question', '')
    prompt = (
        "You are a PhD-level scientist. Answer the following multiple-choice question.\n"
        "Think step by step. End your answer with exactly: 'The answer is (X)' "
        "where X is A, B, C, or D.\n\n"
        f"Question: {question}\n\nChoices:\n{choices_str}"
    )
    return prompt, correct_letter

def extract_gpqa_answer(text):
    m = re.search(r'[Tt]he answer is\s*\(?([A-D])\)?', text)
    if m: return m.group(1)
    m = re.search(r'\*\*([A-D])\*\*', text)
    if m: return m.group(1)
    m = re.search(r'\b([A-D])\b[^\w]*$', text.strip())
    if m: return m.group(1)
    return ''

def is_correct_gpqa(full_text, correct_letter):
    return extract_gpqa_answer(full_text).upper() == correct_letter.upper()

gpqa_data = load_gpqa()
p0, cl0   = gpqa_prompt_and_answer(gpqa_data[0], 0)
print(f'Example Q: {gpqa_data[0].get("Question","")[:100]}...')
print(f'Correct letter idx=0: {cl0}')


README.md: 0.00B [00:00, ?B/s]

gpqa_diamond.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

Loaded 198 GPQA Diamond problems.
Example Q: Two quantum states with energies E1 and E2 have a lifetime of 10^-9 sec and 10^-8 sec, respectively....
Correct letter idx=0: A


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
for line in result.stdout.split('\n')[:8]:
    print(line)

Mon May  4 15:48:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


In [ ]:
if os.path.exists(GPQA_RESULTS_PATH):
    print('Results already saved -- skipping model load.')
    model = tokenizer = None
else:
    gpqa_cache = load_cache(GPQA_CACHE_PATH)
    remaining  = [i for i in range(len(gpqa_data))
                  if not gpqa_cache.get(i, {}).get('done')]
    print(f'Cache: {len(gpqa_data)-len(remaining)}/{len(gpqa_data)} done. '
          f'Remaining: {len(remaining)}')
    if remaining:
        model, tokenizer = load_model(MODEL_ID, quantize_4bit=QUANTIZE)
    else:
        print('All samples cached -- skipping model load.')
        model = tokenizer = None


Cache: 0/198 done. Remaining: 198
Loading tokenizer: Qwen/Qwen2.5-72B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/963 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 462.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 200.81 MiB is free. Including non-PyTorch memory, this process has 79.04 GiB memory in use. Of the allocated memory 78.43 GiB is allocated by PyTorch, and 135.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
if os.path.exists(GPQA_RESULTS_PATH):
    print('Results already exist -- skipping inference.')
    gpqa_cache = load_cache(GPQA_CACHE_PATH)
else:
    gpqa_cache = load_cache(GPQA_CACHE_PATH)
    remaining  = [i for i in range(len(gpqa_data))
                  if not gpqa_cache.get(i, {}).get('done')]

    if model is not None and remaining:
        for i in tqdm(remaining, desc='GPQA-72B'):
            try:
                row    = gpqa_data[i]
                prompt, correct_letter = gpqa_prompt_and_answer(row, i)
                full_text, all_ents   = generate_full(
                    model, tokenizer, prompt,
                    temperature=TEMP, max_new_tokens=MAX_NEW)
                correct    = is_correct_gpqa(full_text, correct_letter)
                has_answer = extract_gpqa_answer(full_text) != ''
                gpqa_cache[i] = {
                    'done': True, 'full_text': full_text,
                    'all_entropies': all_ents, 'correct': correct,
                    'has_answer': has_answer, 'correct_letter': correct_letter,
                }
            except Exception as ex:
                print(f'  Error idx={i}: {ex}')
                gpqa_cache[i] = {'done': False}
            if i % 10 == 0:
                save_cache(gpqa_cache, GPQA_CACHE_PATH)
            free_memory()

        save_cache(gpqa_cache, GPQA_CACHE_PATH)
        del model, tokenizer; free_memory()

n_done    = sum(1 for v in gpqa_cache.values() if isinstance(v, dict) and v.get('done'))
n_correct = sum(1 for v in gpqa_cache.values()
                if isinstance(v, dict) and v.get('done') and v.get('correct'))
n_fmt     = sum(1 for v in gpqa_cache.values()
                if isinstance(v, dict) and v.get('done') and v.get('has_answer'))
print(f'Inference: {n_done}/{len(gpqa_data)} done')
print(f'Format OK: {n_fmt} ({n_fmt/max(n_done,1):.1%})')
print(f'Correct:   {n_correct} ({n_correct/max(n_done,1):.1%})  (expected ~65%)')


In [ ]:
gpqa_cache = load_cache(GPQA_CACHE_PATH)
usable = [v for v in gpqa_cache.values()
          if isinstance(v, dict) and v.get('done')
          and v.get('all_entropies')
          and len(v['all_entropies']) >= 8]

gpqa_labels   = np.array([int(v['correct']) for v in usable])
gpqa_raw_ents = [v['all_entropies'] for v in usable]
gpqa_n_toks   = np.array([len(e) for e in gpqa_raw_ents])

gpqa_feat_list = [extract_all_features(e) for e in gpqa_raw_ents]
valid_mask     = [f is not None for f in gpqa_feat_list]
gpqa_labels    = gpqa_labels[valid_mask]
gpqa_raw_ents  = [e for e, m in zip(gpqa_raw_ents, valid_mask) if m]
gpqa_n_toks    = gpqa_n_toks[valid_mask]
gpqa_feat_list = [f for f in gpqa_feat_list if f is not None]
gpqa_feat_arrays = {fn: np.array([f[fn] for f in gpqa_feat_list]) for fn in FEAT_NAMES}

print(f'GPQA usable: {len(gpqa_labels)} samples | '
      f'{int(gpqa_labels.sum())} correct ({gpqa_labels.mean():.1%})')
print(f'Avg trace:   {gpqa_n_toks.mean():.1f} tok  '
      f'(min={gpqa_n_toks.min()}, max={gpqa_n_toks.max()})')
print(f'Traces < 32: {(gpqa_n_toks < 32).sum()} (STFT zeros)')


In [ ]:
WINDOW_SIZES = [3, 5, 7, 9, 16]

print(f'Window ablation  (n={len(gpqa_labels)}, avg trace {gpqa_n_toks.mean():.1f} tok)')
print(f'{"Window":>8}  {"AUC":>8}  {"95% CI":>16}  {"Eligible":>10}')
print('-' * 50)

gpqa_ablation = []
for w in WINDOW_SIZES:
    sw_vals = np.array([sw_var_peak_with_window(e, w) for e in gpqa_raw_ents])
    n_elig  = sum(1 for e in gpqa_raw_ents if len(e) >= w)
    ap, lop, hip = boot_auc(gpqa_labels,  sw_vals)
    an, lon, hin = boot_auc(gpqa_labels, -sw_vals)
    if ap >= an: auc, lo, hi, sign = ap, lop, hip, '+'
    else:        auc, lo, hi, sign = an, lon, hin, '-'
    gpqa_ablation.append({'w': w, 'auc': auc, 'lo': lo, 'hi': hi,
                          'sign': sign, 'vals': sw_vals})
    print(f'  w={w:<4}  {100*auc:>6.1f}%  [{100*lo:>5.1f}, {100*hi:>5.1f}]  '
          f'{n_elig}/{len(gpqa_raw_ents)}')

gpqa_best_w = max(gpqa_ablation, key=lambda x: x['auc'])
print(f'\nBest window: w={gpqa_best_w["w"]}  AUC={100*gpqa_best_w["auc"]:.1f}%')
gpqa_feat_arrays['sw_var_peak'] = (
    gpqa_best_w['vals'] * (1 if gpqa_best_w['sign'] == '+' else -1))
print(f'gpqa_feat_arrays["sw_var_peak"] updated -> w={gpqa_best_w["w"]}')


In [ ]:
gpqa_auc_map, gpqa_sign_map, rows_gpqa = {}, {}, []
for fn in FEAT_NAMES:
    ap, lop, hip = boot_auc(gpqa_labels,  gpqa_feat_arrays[fn])
    an, lon, hin = boot_auc(gpqa_labels, -gpqa_feat_arrays[fn])
    if ap >= an: gpqa_auc_map[fn], gpqa_sign_map[fn] = ap, +1
    else:        gpqa_auc_map[fn], gpqa_sign_map[fn] = an, -1
    rows_gpqa.append((gpqa_auc_map[fn], fn,
                      lop if ap >= an else lon,
                      hip if ap >= an else hin))
rows_gpqa.sort(reverse=True)

print('Individual Feature AUCs -- GPQA Diamond / Qwen2.5-72B T=1.0')
print(f'{"Signal":<26}  {"AUC":>8}  {"95% CI":>16}  {"sign":>5}')
print('-' * 62)
for auc, name, lo, hi in rows_gpqa:
    flag = ' <- improved vs 7B' if gpqa_auc_map[name] > 0.60 else ''
    print(f'  {name:<26} {100*auc:>7.1f}%  [{100*lo:>5.1f}, {100*hi:>5.1f}]  '
          f'{gpqa_sign_map[name]:>+4d}{flag}')

best_gpqa_ind = max(gpqa_auc_map.values())
print(f'\nBest individual AUC: {100*best_gpqa_ind:.1f}%')
print(f'Prior 7B individual AUCs: ~51-60%')


In [ ]:
print('Running normalized Nadler fusion for GPQA ...')
gpqa_fuse_auc, gpqa_fuse_lo, gpqa_fuse_hi, gpqa_best_subset, gpqa_best_weights = \
    best_nadler_on(gpqa_feat_arrays, FEAT_NAMES, gpqa_labels,
                   max_size=4, label='GPQA-72B', normalize=True, compare_mean=True)

gpqa_nadler_lift_pp = 0.0
if gpqa_best_subset:
    gpqa_oriented_z = {fn: zscore(gpqa_feat_arrays[fn] * gpqa_sign_map[fn])
                       for fn in FEAT_NAMES}
    gpqa_avg_fused, _ = simple_average_fusion(
        *[gpqa_oriented_z[fn] for fn in gpqa_best_subset])
    gpqa_avg_auc, _, _ = boot_auc(gpqa_labels, gpqa_avg_fused)
    gpqa_nadler_lift_pp = (gpqa_fuse_auc - gpqa_avg_auc) * 100

delta_vs_prior = (gpqa_fuse_auc - PRIOR_GPQA_7B) * 100

print()
print('GPQA Diamond -- Qwen2.5-72B-Instruct, T=1.0')
print('-' * 50)
print(f'Our normalized Nadler:   {100*gpqa_fuse_auc:.1f}% '
      f'[{100*gpqa_fuse_lo:.1f}, {100*gpqa_fuse_hi:.1f}]')
print(f'Prior best (Mistral-7B): {100*PRIOR_GPQA_7B:.1f}%')
print(f'Delta vs prior:          {delta_vs_prior:+.1f} pp')
if gpqa_best_subset:
    print(f'Best subset:             {" + ".join(gpqa_best_subset)}')
print('-' * 50)


In [ ]:
g0 = len(gpqa_labels) >= 150
g1 = 0.50 <= gpqa_labels.mean() <= 0.80
g2 = best_gpqa_ind  > 0.57
g3 = gpqa_fuse_auc  > PRIOR_GPQA_7B
g4 = gpqa_fuse_auc  > 0.72
g5 = gpqa_fuse_lo   > 0.60
g6 = gpqa_nadler_lift_pp > 0

gates_b = [
    ('G0', 'Sufficient samples (>=150)',
     g0, f'len={len(gpqa_labels)} >= 150',
         f'len={len(gpqa_labels)} < 150'),
    ('G1', 'Accuracy in sweet spot [50%-80%]',
     g1, f'accuracy={gpqa_labels.mean():.1%} in [50%, 80%]',
         f'Accuracy {gpqa_labels.mean():.1%} out of range'),
    ('G2', 'Spectral structure (ind. AUC > 57%)',
     g2, f'best individual AUC={100*best_gpqa_ind:.1f}% > 57%',
         f'best individual={100*best_gpqa_ind:.1f}%'),
    ('G3', 'Beat prior GPQA best (65.4%)',
     g3, f'fusion={100*gpqa_fuse_auc:.1f}% > prior {100*PRIOR_GPQA_7B:.1f}%',
         f'fusion={100*gpqa_fuse_auc:.1f}% <= prior {100*PRIOR_GPQA_7B:.1f}%'),
    ('G4', 'Strong result: fusion AUC > 72%',
     g4, f'fusion={100*gpqa_fuse_auc:.1f}% > 72%',
         f'fusion={100*gpqa_fuse_auc:.1f}% <= 72%'),
    ('G5', 'Statistically reliable (CI lower > 60%)',
     g5, f'95% CI lower={100*gpqa_fuse_lo:.1f}% > 60%',
         f'95% CI lower={100*gpqa_fuse_lo:.1f}% <= 60%'),
    ('G6', 'Nadler lift over average > 0',
     g6, f'Nadler lift={gpqa_nadler_lift_pp:+.1f}pp > 0',
         f'Nadler lift={gpqa_nadler_lift_pp:.1f}pp'),
]

print('PART B DECISION GATES')
print('=' * 85)
n_pass = 0
for gate_id, name, passed, pass_msg, fail_msg in gates_b:
    status = 'PASS' if passed else 'FAIL'
    msg    = pass_msg if passed else fail_msg
    print(f'{gate_id}  {name:<42}  [{status}]  {msg}')
    if passed: n_pass += 1
print(f'\n{n_pass}/{len(gates_b)} Part B gates passed.')
print()
if g4:   print('-> Spectral features generalize to graduate-level science MCQ.')
elif g3: print('-> Spectral features transfer with 72B. Not as strong as math.')
elif g2: print('-> Marginal signal. 72B helps vs 7B but domain gap remains.')
else:    print('-> No spectral signal even with 72B.')


In [ ]:
COLORS = {'our_norm': '#2166ac', 'baseline': '#aaaaaa'}

# ── Plot B1: Feature AUCs 72B vs prior 7B ─────────────────────────────────────
prior_7b = {fn: 0.555 for fn in FEAT_NAMES}
prior_7b.update({'dominant_freq': 0.595, 'stft_max_high_power': 0.590,
                 'spectral_entropy': 0.560, 'epr': 0.545})

fn_b1 = sorted(FEAT_NAMES, key=lambda fn: gpqa_auc_map[fn], reverse=True)
x_b1  = np.arange(len(fn_b1)); w_bar = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_b1 - w_bar/2, [100*gpqa_auc_map[fn] for fn in fn_b1],
       w_bar, label='Qwen2.5-72B (this run)', color=COLORS['our_norm'], alpha=0.85)
ax.bar(x_b1 + w_bar/2, [100*prior_7b[fn] for fn in fn_b1],
       w_bar, label='Prior Mistral-7B (~Phase 5)', color=COLORS['baseline'], alpha=0.7)
ax.axhline(50,                color='black',  linestyle='--', linewidth=1, label='Chance (50%)')
ax.axhline(100*PRIOR_GPQA_7B, color='orange', linestyle='-.', linewidth=1.5,
           label=f'Prior best fusion 7B ({100*PRIOR_GPQA_7B:.1f}%)')
ax.set_xticks(x_b1)
ax.set_xticklabels([fn.replace('_', '\n') for fn in fn_b1], fontsize=8)
ax.set_ylabel('AUC (%)'); ax.legend(fontsize=9)
ax.set_title('GPQA Diamond Feature AUCs: 72B vs prior 7B')
plt.tight_layout()
plt.savefig(GPQA_PLOT_DIR + 'B1_gpqa_feature_aucs_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B2: Full results landscape ───────────────────────────────────────────
landscape_data = [
    ('MATH-500/Qwen-7B/T=1.5',   96.6, 'math'),
    ('MATH-500/Qwen-7B/T=1.0',   90.0, 'math'),
    ('MATH-500/Qwen-1.5B/T=1.5', 88.3, 'math'),
    ('GSM8K/Llama-8B',           76.0, 'math'),
    ('GPQA/Qwen-72B (this run)', 100*gpqa_fuse_auc, 'science'),
    ('GPQA/Mistral-7B',          65.4, 'science'),
    ('HotpotQA/Mistral-7B',      59.5, 'factual'),
]
landscape_data.sort(key=lambda x: x[1], reverse=True)
type_colors = {'math': '#4dac26', 'science': '#f4a582', 'factual': '#d73027'}

fig, ax = plt.subplots(figsize=(11, 5))
lbl_names = [d[0] for d in landscape_data]
lbl_aucs  = [d[1] for d in landscape_data]
lbl_cols  = [type_colors[d[2]] for d in landscape_data]
bars = ax.barh(lbl_names, lbl_aucs, color=lbl_cols, edgecolor='white', linewidth=0.8)
ax.axvline(50, color='black', linestyle='--', linewidth=1)
for bar, v in zip(bars, lbl_aucs):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=9)
legend_patches = [mpatches.Patch(color=c, label=t.title())
                  for t, c in type_colors.items()]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_xlabel('AUROC (%)'); ax.set_xlim(45, 102)
ax.set_title('Full Results Landscape -- Spectral Hallucination Detection')
plt.tight_layout()
plt.savefig(GPQA_PLOT_DIR + 'B2_full_results_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B3: GPQA H(n) trajectories (optional) ────────────────────────────────
gpqa_correct_idx   = [i for i, v in enumerate(gpqa_labels)
                      if v == 1 and len(gpqa_raw_ents[i]) > 100]
gpqa_incorrect_idx = [i for i, v in enumerate(gpqa_labels)
                      if v == 0 and len(gpqa_raw_ents[i]) > 100]

if gpqa_correct_idx and gpqa_incorrect_idx:
    rng_b3 = np.random.default_rng(11)
    sel_gc = rng_b3.choice(gpqa_correct_idx,   size=min(4, len(gpqa_correct_idx)),   replace=False)
    sel_gw = rng_b3.choice(gpqa_incorrect_idx, size=min(4, len(gpqa_incorrect_idx)), replace=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, idxs, color, label_txt in [
        (axes[0], sel_gc, 'green', 'Correct'),
        (axes[1], sel_gw, 'red',   'Incorrect'),
    ]:
        traces = [np.array(gpqa_raw_ents[i]) for i in idxs]
        for t in traces:
            ax.plot(t, alpha=0.25, linewidth=0.8, color=color)
        if traces:
            min_len = min(len(t) for t in traces)
            mean_tr = np.mean([t[:min_len] for t in traces], axis=0)
            ax.plot(mean_tr, linewidth=2.2, color=color, label=f'Mean {label_txt}')
        ax.set_xlabel('Token position'); ax.set_ylabel('H(n) entropy')
        ax.set_title(f'{label_txt} (n={len(idxs)})')
        ax.legend(fontsize=8)
    fig.suptitle('GPQA Entropy Trajectories -- Qwen2.5-72B', fontsize=12)
    plt.tight_layout()
    plt.savefig(GPQA_PLOT_DIR + 'B3_gpqa_trajectories.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping B3: not enough long traces (>100 tok) in both classes.')


In [ ]:
gpqa_results = {
    'phase':             8,
    'part':              'B',
    'model_id':          MODEL_ID,
    'dataset':           'gpqa_diamond',
    'temp':              TEMP,
    'n_samples':         int(len(gpqa_labels)),
    'accuracy':          float(gpqa_labels.mean()),
    'avg_trace':         float(gpqa_n_toks.mean()),
    'feat_names':        FEAT_NAMES,
    'raw_labels':        gpqa_labels.tolist(),
    'feat_arrays':       {k: v.tolist() for k, v in gpqa_feat_arrays.items()},
    'auc_map':           {k: float(v) for k, v in gpqa_auc_map.items()},
    'sign_map':          {k: int(v)   for k, v in gpqa_sign_map.items()},
    'ablation':          [{'w': r['w'], 'auc': float(r['auc']),
                            'lo': float(r['lo']), 'hi': float(r['hi'])}
                          for r in gpqa_ablation],
    'best_window':       int(gpqa_best_w['w']),
    'fusion_auc':        float(gpqa_fuse_auc),
    'fusion_lo':         float(gpqa_fuse_lo),
    'fusion_hi':         float(gpqa_fuse_hi),
    'best_subset':       list(gpqa_best_subset) if gpqa_best_subset else [],
    'nadler_lift_pp':    float(gpqa_nadler_lift_pp),
    'prior_gpqa_7b_auc': PRIOR_GPQA_7B,
    'delta_vs_prior':    float(gpqa_fuse_auc - PRIOR_GPQA_7B),
    'gates_b':           {g[0]: bool(g[2]) for g in gates_b},
    'n_pass_b':          n_pass,
}
save_cache(gpqa_results, GPQA_RESULTS_PATH)

print('=' * 52)
print('GPQA Diamond (Qwen2.5-72B) -- Final Results')
print('=' * 52)
print(f'Samples:         {len(gpqa_labels)}')
print(f'Accuracy:        {gpqa_labels.mean():.1%}')
print(f'Fusion AUC:      {100*gpqa_fuse_auc:.1f}% '
      f'[{100*gpqa_fuse_lo:.1f}, {100*gpqa_fuse_hi:.1f}]')
print(f'Best subset:     {" + ".join(gpqa_best_subset) if gpqa_best_subset else "n/a"}')
print(f'Prior 7B best:   {100*PRIOR_GPQA_7B:.1f}%')
print(f'Delta:           {(gpqa_fuse_auc-PRIOR_GPQA_7B)*100:+.1f} pp')
print(f'Nadler lift:     {gpqa_nadler_lift_pp:+.1f} pp')
print(f'Gates passed:    {n_pass}/{len(gates_b)}')
print('=' * 52)
print(f'\nSaved: {GPQA_RESULTS_PATH}')
